# 🎯 M4 + M5 · Tracking, Demo & Reproducibility  (LOCAL / Windows dev version)

Runs locally against the unzipped project folder — no `drive.mount`, no Colab installs.
Device is auto-detected; a CUDA gate up top fails loudly if the GPU isn't actually usable.

> **Submission note:** the brief requires a *Colab* notebook. When merging for submission,
> re-add the Colab mount + paths and run-all on a fresh Colab runtime. This file is for local dev.


## ⚙️ 0 · Environment & GPU Check

Dependencies are managed by your `.venv` (installed via `uv`), so there is **no** `pip install`
cell here. If the GPU check below fails, fix the environment before running anything else —
do not run on CPU by accident.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# Hard gate: catch the two RTX 50-series failure modes explicitly.
assert torch.cuda.is_available(), (
    "CUDA not available — you are on the CPU build. Reinstall into THIS venv:\n"
    "  uv pip install --python .venv\\Scripts\\python.exe torch torchvision "
    "--index-url https://download.pytorch.org/whl/cu128 --force-reinstall\n"
    "then RESTART THE KERNEL (cell re-run is not enough)."
)

cap = torch.cuda.get_device_capability(0)
print("device:", torch.cuda.get_device_name(0), "| compute capability:", cap)

# Real kernel test — is_available() can be True while kernels are missing for sm_120.
x = torch.randn(2000, 2000, device="cuda")
_ = (x @ x).sum().item()
print("✅ GPU matmul executed — Blackwell kernels present. Good to go.")

DEVICE = 0  # YOLO device id; gate above guarantees CUDA is real


## 📁 1 · Paths (local)

Set `PROJECT_ROOT` to the folder that contains `data/`, `models/`, `VIDEOS/`.
The cell auto-detects it by searching for `data/processed/data.yaml`; override manually if needed.


In [ ]:
import os, glob
from pathlib import Path

# --- OPTION A: let it auto-detect (searches upward + common spots) ---
def _find_root(start='.'):
    here = Path(start).resolve()
    candidates = [here] + list(here.parents) + list(here.glob('**/'))
    for c in candidates:
        if (c / 'data' / 'processed' / 'data.yaml').exists():
            return c
    return None

_auto = _find_root('.')

# --- OPTION B: hardcode if auto-detect misses (raw string, note the leading r) ---
# PROJECT_ROOT = Path(r'C:\Users\vldma\Comp vision\Counter_UAV_project\!CVIS')

PROJECT_ROOT = _auto  # or replace with Option B
assert PROJECT_ROOT is not None, (
    'Could not locate the project root (the folder with data/processed/data.yaml). '
    'Set PROJECT_ROOT manually via Option B above.')

DATA_DIR   = PROJECT_ROOT / 'data' / 'processed'
DATA_YAML  = str(DATA_DIR / 'data.yaml')
MODELS_DIR = PROJECT_ROOT / 'models'
RUNS_DIR   = str(PROJECT_ROOT / 'runs')
VIDEOS_DIR = PROJECT_ROOT / 'VIDEOS'

# Nano beat Small on validation (mAP50 0.945 vs 0.926) -> default demo model.
PRIMARY_WEIGHTS = str(MODELS_DIR / 'best_n.pt')
assert os.path.exists(PRIMARY_WEIGHTS), f'missing weights: {PRIMARY_WEIGHTS}'

print('root   :', PROJECT_ROOT)
print('weights:', PRIMARY_WEIGHTS)
print('videos :', len(glob.glob(str(VIDEOS_DIR / '*.mp4'))), 'clips')


In [ ]:
import yaml
with open(DATA_YAML) as f:
    d = yaml.safe_load(f)
d['path'] = str(DATA_DIR).replace('\\', '/')
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(d, f, sort_keys=False)
print("data.yaml path ->", d['path'])

## ✅ A · Test-Set Evaluation — REQUIRED (brief item 5)

Training logs *validation* metrics; the brief asks for **test** metrics. This runs the held-out
test split for both model scales so the comparison is on data neither model saw.


In [ ]:
from ultralytics import YOLO
import pandas as pd
from pathlib import Path

rows = []
for tag in ['n', 's']:
    w = str(MODELS_DIR / f'best_{tag}.pt')
    m = YOLO(w)
    r = m.val(data=DATA_YAML, split='test', device=DEVICE,
              project=RUNS_DIR, name=f'test_eval_{tag}', exist_ok=True, verbose=False)
    rows.append({'model': f'YOLOv11{tag}',
                 'mAP50': round(r.box.map50, 4), 'mAP50-95': round(r.box.map, 4),
                 'precision': round(r.box.mp, 4), 'recall': round(r.box.mr, 4)})

test_df = pd.DataFrame(rows)
test_df.to_csv(str(Path(RUNS_DIR) / 'test_set_comparison.csv'), index=False)

print('\n' + '='*52)
print('  TEST-SET COMPARISON  (this is the deck slide)')
print('='*52)
print(test_df.to_string(index=False))
print('='*52)
# which model wins on the headline metric?
win = test_df.loc[test_df['mAP50-95'].idxmax(), 'model']
print(f'Higher mAP50-95: {win}')
try:
    from IPython.display import display
    display(test_df)
except Exception:
    pass


In [ ]:
# Per-class test AP (drone vs bird) for the primary model -> failure-analysis slide.
m = YOLO(PRIMARY_WEIGHTS)
r = m.val(data=DATA_YAML, split='test', device=DEVICE,
          project=RUNS_DIR, name='test_eval_perclass', exist_ok=True, verbose=False)
for i, name in r.names.items():
    try:    print(f'{name:6} test AP50 = {r.box.ap50[i]:.4f}')
    except Exception: print(f'{name:6} (no test instances)')
print('\nConfusion matrix saved under:', r.save_dir)


## 🎯 B · Object Tracking  (committed bonus)

`model.track()` + ByteTrack: persistent IDs across frames in one call, writes an annotated mp4.
On your RTX 5080 the full-res clips track comfortably — no need to trim.


In [ ]:
for p in sorted(glob.glob(str(VIDEOS_DIR / '*.mp4'))):
    print(os.path.basename(p))


In [ ]:
from ultralytics import YOLO

CLIP = str(VIDEOS_DIR / '11498840-hd_1920_1080_30fps.mp4')   # large foreground drone

model = YOLO(PRIMARY_WEIGHTS)
# stream=True + consuming the generator avoids the RAM-accumulation warning;
# save=True still writes the annotated video frame-by-frame.
gen = model.track(
    source=CLIP, tracker='bytetrack.yaml', conf=0.25, iou=0.5,
    persist=True, save=True, device=DEVICE,
    project=RUNS_DIR, name='track_demo', exist_ok=True, stream=True, verbose=False)
for _ in gen:
    pass
print('Annotated tracking video written under:', str(Path(RUNS_DIR) / 'track_demo'))


In [ ]:
# Re-encode the tracking output to a browser-playable .mp4 (ultralytics often writes .avi).
# Dependency-free: uses OpenCV, already installed.
import glob, cv2, os
from pathlib import Path

src_dir = str(Path(RUNS_DIR) / 'track_demo')
avis = glob.glob(os.path.join(src_dir, '*.avi'))
made = []
for avi in avis:
    mp4 = os.path.splitext(avi)[0] + '_playable.mp4'
    cap = cv2.VideoCapture(avi)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # 'avc1' (H.264) plays inline in browsers/Colab; falls back to 'mp4v' if unavailable.
    for cc in ('avc1', 'mp4v'):
        writer = cv2.VideoWriter(mp4, cv2.VideoWriter_fourcc(*cc), fps, (w, h))
        if writer.isOpened():
            break
    n = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        writer.write(fr); n += 1
    cap.release(); writer.release()
    made.append(mp4)
    print(f'Re-encoded -> {mp4}  ({n} frames)')
if not made:
    print('No .avi found — output may already be .mp4.')


### B.1 · Tracking stability (rigorous, not a gimmick)

No ground-truth track IDs -> MOTA/IDF1 can't be computed. Report a defensible proxy:
unique IDs assigned vs. true object count. Excess unique IDs on a single-object clip = ID switches.


In [ ]:
import numpy as np
model = YOLO(PRIMARY_WEIGHTS)
ids_seen, per_frame = set(), []
for r in model.track(source=CLIP, tracker='bytetrack.yaml', persist=True,
                     stream=True, conf=0.25, device=DEVICE, verbose=False):
    if r.boxes is not None and r.boxes.id is not None:
        ids = r.boxes.id.int().tolist()
        ids_seen.update(ids); per_frame.append(len(ids))
print(f'Frames with detections : {len(per_frame)}')
print(f'Unique track IDs        : {len(ids_seen)}')
print(f'Avg objects per frame   : {np.mean(per_frame):.2f}' if per_frame else 'no detections')
print('\nIf the clip has 1 drone, unique-IDs minus 1 = ID-switch count.')


## 🖥️ C · Reusable Inference (live-demo path)

One function the live demo calls; mirrors the standalone `infer.py`.


In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

def run_inference(source, weights=PRIMARY_WEIGHTS, conf=0.25, save=True, name='demo_infer'):
    model = YOLO(weights)
    return model.predict(source=source, conf=conf, save=save, device=DEVICE,
                         project=RUNS_DIR, name=name, exist_ok=True, verbose=False)

sample_imgs = sorted(glob.glob(str(DATA_DIR / 'test' / 'images' / '*.jpg')))[:3]
res = run_inference(sample_imgs, name='demo_infer_imgs')
for p in glob.glob(str(Path(RUNS_DIR) / 'demo_infer_imgs' / '*.jpg'))[:3]:
    display(IPImage(filename=p, width=520))


## 🎬 D · Recorded Fallback (bank this BEFORE the presentation)

Live demo is the gamble; the recorded mp4 is what gets shown if anything stalls.
Section B already produced one — this cell locates and plays it back.


In [ ]:
import glob, base64
from IPython.display import HTML
from pathlib import Path

d = str(Path(RUNS_DIR) / 'track_demo')
# prefer the re-encoded playable mp4, then any mp4, then avi
cands = (sorted(glob.glob(d + '/*_playable.mp4')) +
         sorted(glob.glob(d + '/*.mp4')) +
         sorted(glob.glob(d + '/*.avi')))
assert cands, 'No annotated video found — run the tracking + re-encode cells first.'
vid = cands[0]
print('Fallback video:', vid)
if vid.endswith('.mp4'):
    data = base64.b64encode(open(vid,'rb').read()).decode()
    display(HTML(f'<video width=560 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
else:
    print('Only .avi available — open it directly; run the re-encode cell to get an inline-playable mp4.')


## 🧩 E · (Optional) SAHI — small distant objects

Only for the **bird** clips with tiny distant specks; skip for drone footage (drone fills the frame).
Tiling raises small-object recall at a large FPS cost — state that tradeoff if you present it.


In [ ]:
!uv pip install sahi
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import cv2
det = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path=PRIMARY_WEIGHTS,
    confidence_threshold=0.25, device='cuda:0')
cap = cv2.VideoCapture(str(VIDEOS_DIR / '13689813-uhd_2160_3840_60fps.mp4'))
cap.set(cv2.CAP_PROP_POS_FRAMES, 800); ok, frame = cap.read(); cap.release()
res = get_sliced_prediction(frame, det, slice_height=512, slice_width=512,
                             overlap_height_ratio=0.2, overlap_width_ratio=0.2)
print('SAHI detections:', len(res.object_prediction_list))


## 🔁 F · Reproducibility

1. Deps pinned in your `.venv` (torch from the cu128 index for the RTX 5080).
2. Training used a fixed seed (`seed=42`).
3. **Before submitting**, port to Colab and do a fresh-runtime run-all there — that is the graded path.
4. Confirm `data.yaml` `path:` matches wherever the data lives in the submission environment.
